# Gold: Earthquake Daily Summary (v1) — SQL

SQL-cell equivalent of `gold_earthquake_daily_summary_v1.ipynb`, using
`%%sql` magic instead of the DataFrame API. Same result: reads the
`earthquake_data.earthquakes` bronze table and persists
`earthquake_data.gold_earthquake_daily_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/bronze_ingest_usgs_earthquakes.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: daily earthquake summary

`CREATE OR REPLACE TABLE` keeps this re-runnable.

In [ ]:
%%sql
CREATE OR REPLACE TABLE earthquake_data.gold_earthquake_daily_summary_v1 AS
SELECT
    CAST(event_time AS DATE) AS event_date,
    COUNT(*) AS total_quakes,
    ROUND(AVG(magnitude), 2) AS avg_magnitude,
    ROUND(MAX(magnitude), 2) AS max_magnitude,
    SUM(CASE WHEN magnitude >= 4.0 THEN 1 ELSE 0 END) AS quakes_mag4_plus
FROM earthquake_data.earthquakes
GROUP BY CAST(event_time AS DATE)
ORDER BY event_date

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_days FROM earthquake_data.gold_earthquake_daily_summary_v1

In [ ]:
%%sql
SELECT * FROM earthquake_data.gold_earthquake_daily_summary_v1
ORDER BY event_date DESC
LIMIT 10